In [1]:
# Setup: run this first.
# In Databricks, `spark` already exists and this cell only sets CATALOG.
# In VS Code, it connects to Databricks serverless through Databricks Connect.
import glob, os, shutil

CATALOG = "hertz"

try:
    spark
except NameError:
    # Login uses the `databricks` CLI. If it isn't on PATH, use the copy bundled with the VS Code extension.
    if not shutil.which("databricks"):
        bundled = sorted(glob.glob(os.path.expanduser(r"~\.vscode\extensions\databricks.databricks-*\bin")))
        if bundled:
            os.environ["PATH"] = bundled[-1] + os.pathsep + os.environ["PATH"]
    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "vscode")

    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

print("Connected. Spark version:", spark.version)

Connected. Spark version: 4.2.0


# Bronze: load the raw CSVs into tables
Each CSV in `/Volumes/hertz/bronze/raw` becomes one `hertz.bronze.*` table, **unchanged**. Every column is kept as a string, and two lineage columns are added. Bronze is never cleaned, so silver can always be rebuilt from it.

In [0]:
from pyspark.sql import functions as F

CATALOG = "hertz"
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw"
DATASETS = ["job_types", "rental_demand", "vehicle_classes", "work_orders"]

for name in DATASETS:
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)          # bronze = raw strings; types are set in silver
          .csv(f"{VOLUME_PATH}/{name}.csv")
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingested_at", F.current_timestamp()))

    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.bronze.{name}"))
    print(f"{CATALOG}.bronze.{name}: {df.count():,} rows")

hertz.bronze.job_types: 7 rows
hertz.bronze.rental_demand: 42 rows
hertz.bronze.vehicle_classes: 6 rows
hertz.bronze.work_orders: 65 rows


# Silver: clean each table
Reads from `hertz.bronze.*` and writes `hertz.silver.*` tables.

**Cleaning rules**
1. **Types**: dates, timestamps, numbers and booleans are cast explicitly, so nothing depends on schema inference.
2. **Categorical values** (`vehicle_class`, `job_type`, `status`) are trimmed and lowercased, so joins match across tables (`battery` = `Battery`). ID columns are left as-is.
3. **Exact duplicate rows** are dropped (WO-054, WO-024, WO-007).
4. **Open/Closed conflicts**: when the same vehicle, job and `opened_ts` appear as both `open` and `closed`, the job is done. The stale `open` row is dropped (WO-015, WO-031, WO-008, WO-058). *Assumption: a Closed record is the newer truth.*
5. **Closed rows are kept** in silver, so the history stays complete. The gold layer filters to `status = 'open'` for planning.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

def bronze(name):
    return spark.table(f"{CATALOG}.bronze.{name}")

def category(col):
    """Normalize a categorical column so it joins consistently across tables."""
    return F.lower(F.trim(F.col(col))).alias(col)

def save_silver(df, name):
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.silver.{name}"))
    print(f"{CATALOG}.silver.{name}: {df.count():,} rows")

# --- job_types ---------------------------------------------------------------
job_types = bronze("job_types").select(
    category("job_type"),
    F.col("shop_hours").cast("double").alias("shop_hours"),      # NULL for vendor-only jobs
    F.col("parts_cost").cast("double").alias("parts_cost"),
    (F.upper(F.trim("in_house")) == "Y").alias("in_house"),     # Y/N -> boolean
    F.col("vendor_price").cast("double").alias("vendor_price"),
    F.col("vendor_days").cast("int").alias("vendor_days"),
).dropDuplicates()
save_silver(job_types, "job_types")

# --- vehicle_classes ---------------------------------------------------------
vehicle_classes = bronze("vehicle_classes").select(
    category("vehicle_class"),
    F.col("daily_rate").cast("double").alias("daily_rate"),
).dropDuplicates()
save_silver(vehicle_classes, "vehicle_classes")

# --- rental_demand -----------------------------------------------------------
rental_demand = bronze("rental_demand").select(
    F.to_date("date").alias("date"),
    category("vehicle_class"),
    F.col("reservations").cast("int").alias("reservations"),
    F.col("cars_ready").cast("int").alias("cars_ready"),
).dropDuplicates()
save_silver(rental_demand, "rental_demand")

# --- work_orders -------------------------------------------------------------
wo = bronze("work_orders").select(
    F.trim("work_order_id").alias("work_order_id"),
    F.trim("vehicle_id").alias("vehicle_id"),
    category("vehicle_class"),
    category("job_type"),
    F.to_timestamp(F.trim("opened_ts"), "yyyy-MM-dd HH:mm").alias("opened_ts"),
    category("status"),
)
n_raw = wo.count()

# Rule 3: exact duplicates
wo = wo.dropDuplicates()
n_dedup = wo.count()

# Rule 4: drop 'open' rows that have a matching 'closed' row for the same job
job_key = ["vehicle_id", "job_type", "opened_ts"]
closed_jobs = wo.filter("status = 'closed'").select(*job_key)
stale_open = (wo.filter("status = 'open'")
                .join(closed_jobs, job_key, "left_semi")
                .select("work_order_id"))
print("Stale open rows dropped:", sorted(r.work_order_id for r in stale_open.collect()))
work_orders = wo.join(stale_open, "work_order_id", "left_anti")

print(f"work_orders: {n_raw} raw -> {n_dedup} after dedup -> {work_orders.count()} after resolving open/closed conflicts")
save_silver(work_orders, "work_orders")

hertz.silver.job_types: 7 rows
hertz.silver.vehicle_classes: 6 rows
hertz.silver.rental_demand: 42 rows
Stale open rows dropped: ['WO-008', 'WO-015', 'WO-031', 'WO-058']
work_orders: 65 raw -> 62 after dedup -> 58 after resolving open/closed conflicts
hertz.silver.work_orders: 58 rows


# Reviewing the silver tables
Check the schemas, look for leftover spelling or formatting variants, and run the data quality checks.

In [0]:
cleaned_dfs = {name: spark.table(f"{CATALOG}.silver.{name}") for name in DATASETS}

for name, df in cleaned_dfs.items():
    print(f"\n=== {CATALOG}.silver.{name} ===")
    print(f"Rows: {df.count():,}  |  Columns: {len(df.columns)}")
    df.printSchema()
    display(df.limit(20))


=== hertz.silver.job_types ===
Rows: 7  |  Columns: 6
root
 |-- job_type: string (nullable = true)
 |-- shop_hours: double (nullable = true)
 |-- parts_cost: double (nullable = true)
 |-- in_house: boolean (nullable = true)
 |-- vendor_price: double (nullable = true)
 |-- vendor_days: integer (nullable = true)



job_type,shop_hours,parts_cost,in_house,vendor_price,vendor_days
oil change,0.75,45.0,true,195.0,1
collision repair,null,null,false,2260.0,5
tires,1.5,380.0,true,580.0,1
brakes,2.5,260.0,true,540.0,1
transmission,null,null,false,3260.0,7
ac repair,3.0,420.0,true,840.0,2
battery,1.5,210.0,true,400.0,1



=== hertz.silver.rental_demand ===
Rows: 42  |  Columns: 4
root
 |-- date: date (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- reservations: integer (nullable = true)
 |-- cars_ready: integer (nullable = true)



date,vehicle_class,reservations,cars_ready
2026-10-18,economy,40,48
2026-10-12,full-size,15,19
2026-10-12,premium suv,10,10
2026-10-13,full-size,15,19
2026-10-14,full-size,15,19
2026-10-12,suv,24,24
2026-10-14,suv,40,24
2026-10-15,premium suv,15,10
2026-10-16,premium suv,15,10
2026-10-13,economy,40,48



=== hertz.silver.vehicle_classes ===
Rows: 6  |  Columns: 2
root
 |-- vehicle_class: string (nullable = true)
 |-- daily_rate: double (nullable = true)



vehicle_class,daily_rate
full-size,63.0
economy,42.0
premium suv,118.0
midsize,55.0
minivan,92.0
suv,82.0



=== hertz.silver.work_orders ===
Rows: 58  |  Columns: 6
root
 |-- work_order_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- opened_ts: timestamp (nullable = true)
 |-- status: string (nullable = true)



work_order_id,vehicle_id,vehicle_class,job_type,opened_ts,status
WO-029,V-67249,suv,brakes,2026-10-10T22:00:00.000Z,open
WO-022,V-25061,full-size,tires,2026-10-09T12:00:00.000Z,open
WO-062,V-32363,midsize,collision repair,2026-10-08T22:00:00.000Z,closed
WO-047,V-50191,full-size,ac repair,2026-10-11T13:00:00.000Z,open
WO-050,V-28626,midsize,ac repair,2026-10-11T08:00:00.000Z,open
WO-028,V-22738,minivan,tires,2026-10-11T05:00:00.000Z,open
WO-057,V-47131,full-size,collision repair,2026-10-08T22:00:00.000Z,open
WO-054,V-27482,premium suv,ac repair,2026-10-09T20:00:00.000Z,open
WO-037,V-18833,suv,brakes,2026-10-10T16:00:00.000Z,open
WO-025,V-97072,full-size,tires,2026-10-08T17:00:00.000Z,open


In [0]:
# Check for spelling mistakes / inconsistent values in string columns
# Strategy: look at distinct values in low-cardinality string columns (likely
# categorical), and flag values that are suspiciously similar to each other
# (e.g. typos like "sedaan" vs "sedan", or casing/whitespace variants).

import difflib

for dataset_name, df in cleaned_dfs.items():
    print(f"\n==================== {dataset_name} ====================")

    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]

    for c in string_cols:
        distinct_vals = [row[c] for row in df.select(c).distinct().collect() if row[c] is not None]

        # Skip free-text / high-cardinality columns; spelling review only
        # makes sense for a small set of category-like values
        if len(distinct_vals) == 0 or len(distinct_vals) > 30:
            continue

        print(f"\n-- {c} ({len(distinct_vals)} distinct values) --")
        for v in sorted(distinct_vals):
            count = df.filter(F.col(c) == v).count()
            print(f"   {v!r}: {count:,} rows")

        # Flag near-duplicate values that may be typos or formatting variants
        suspects = []
        for i, v1 in enumerate(distinct_vals):
            for v2 in distinct_vals[i + 1:]:
                s1, s2 = str(v1).strip().lower(), str(v2).strip().lower()
                ratio = difflib.SequenceMatcher(None, s1, s2).ratio()
                # Similar but not identical, and not just whitespace/casing copies
                if 0.75 <= ratio < 1.0:
                    suspects.append((v1, v2, ratio))
                # Whitespace or casing-only differences
                elif s1 == s2 and str(v1) != str(v2):
                    suspects.append((v1, v2, 1.0))

        if suspects:
            print("   >>> Possible spelling/formatting issues:")
            for v1, v2, ratio in suspects:
                print(f"       {v1!r}  <->  {v2!r}   (similarity: {ratio:.2f})")



==================== job_types ====================

-- job_type (7 distinct values) --
   'ac repair': 1 rows
   'battery': 1 rows
   'brakes': 1 rows
   'collision repair': 1 rows
   'oil change': 1 rows
   'tires': 1 rows
   'transmission': 1 rows

==================== rental_demand ====================

-- vehicle_class (6 distinct values) --
   'economy': 7 rows
   'full-size': 7 rows
   'midsize': 7 rows
   'minivan': 7 rows
   'premium suv': 7 rows
   'suv': 7 rows

==================== vehicle_classes ====================

-- vehicle_class (6 distinct values) --
   'economy': 1 rows
   'full-size': 1 rows
   'midsize': 1 rows
   'minivan': 1 rows
   'premium suv': 1 rows
   'suv': 1 rows

==================== work_orders ====================

-- vehicle_class (6 distinct values) --
   'economy': 12 rows
   'full-size': 11 rows
   'midsize': 12 rows
   'minivan': 2 rows
   'premium suv': 5 rows
   'suv': 16 rows

-- job_type (7 distinct values) --
   'ac repair': 10 rows
   'ba

In [0]:
# Data quality checks: stop the notebook if silver isn't fit for planning
wo, jt, vc, rd = (cleaned_dfs[n] for n in ["work_orders", "job_types", "vehicle_classes", "rental_demand"])

checks = {
    # every work order ID appears once
    "duplicate work_order_id":
        wo.groupBy("work_order_id").count().filter("count > 1").count(),
    # every job_type / vehicle_class in work orders exists in the lookup tables
    "work_orders.job_type not in job_types":
        wo.join(jt, "job_type", "left_anti").count(),
    "work_orders.vehicle_class not in vehicle_classes":
        wo.join(vc, "vehicle_class", "left_anti").count(),
    "rental_demand.vehicle_class not in vehicle_classes":
        rd.join(vc, "vehicle_class", "left_anti").count(),
    # casts that silently failed become NULL
    "work_orders with NULL opened_ts":
        wo.filter(F.col("opened_ts").isNull()).count(),
    "rental_demand with NULL date":
        rd.filter(F.col("date").isNull()).count(),
    # in-house jobs need shop hours and parts cost to be costed
    "in-house job_types missing shop_hours/parts_cost":
        jt.filter("in_house AND (shop_hours IS NULL OR parts_cost IS NULL)").count(),
}

for name, bad in checks.items():
    print(f"{'OK  ' if bad == 0 else 'FAIL'}  {name}: {bad}")

display(wo.groupBy("status").count())
assert all(v == 0 for v in checks.values()), "Data quality checks failed, see above"

OK    duplicate work_order_id: 0
OK    work_orders.job_type not in job_types: 0
OK    work_orders.vehicle_class not in vehicle_classes: 0
OK    rental_demand.vehicle_class not in vehicle_classes: 0
OK    work_orders with NULL opened_ts: 0
OK    rental_demand with NULL date: 0
OK    in-house job_types missing shop_hours/parts_cost: 0


status,count
open,54
closed,4


In [2]:
# The week at a glance: reserved vs. ready vs. in repair, per class
import pandas as pd

rd = spark.table(f"{CATALOG}.silver.rental_demand").toPandas()
vc = spark.table(f"{CATALOG}.silver.vehicle_classes").toPandas()
jobs = (spark.table(f"{CATALOG}.silver.work_orders").filter("status = 'open'")
        .join(spark.table(f"{CATALOG}.silver.job_types"), "job_type").toPandas())

repairs = (jobs.groupby("vehicle_class")
           .agg(in_repair=("work_order_id", "count"),
                shop_jobs=("in_house", "sum"))
           .assign(vendor_only=lambda d: d.in_repair - d.shop_jobs))

summary = (rd.assign(gap=rd.reservations - rd.cars_ready)
           .groupby("vehicle_class")
           .agg(cars_ready=("cars_ready", "max"),
                peak_reserved=("reservations", "max"),
                peak_short=("gap", "max"),
                days_short=("gap", lambda g: (g > 0).sum()))
           .join(repairs)
           .join(vc.set_index("vehicle_class"))
           .assign(peak_short=lambda d: d.peak_short.clip(lower=0),
                   short_if_all_fixed=lambda d: (d.peak_reserved - d.cars_ready - d.in_repair).clip(lower=0))
           .sort_values("peak_short", ascending=False)
           .reset_index())

display(summary[["vehicle_class", "daily_rate", "cars_ready", "peak_reserved", "peak_short", "days_short",
                 "in_repair", "shop_jobs", "vendor_only", "short_if_all_fixed"]])

# Day-by-day shortage before any repairs (reservations - cars ready, never below 0)
display(rd.assign(short=(rd.reservations - rd.cars_ready).clip(lower=0))
          .pivot(index="vehicle_class", columns="date", values="short")
          .loc[summary.vehicle_class])

,vehicle_class,daily_rate,cars_ready,peak_reserved,peak_short,days_short,in_repair,shop_jobs,vendor_only,short_if_all_fixed
0,suv,82.0,24,40,16,6,15,15,0,1
1,midsize,55.0,43,53,10,6,11,11,0,0
2,premium suv,118.0,10,15,5,6,5,5,0,0
3,minivan,92.0,13,15,2,6,2,2,0,0
4,full-size,63.0,19,15,0,0,10,8,2,0
5,economy,42.0,48,40,0,0,11,10,1,0


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
suv,0,16,16,16,16,16,16
midsize,0,2,4,6,8,10,8
premium suv,0,5,5,5,5,5,5
minivan,0,2,2,2,2,2,2
full-size,0,0,0,0,0,0,0
economy,0,0,0,0,0,0,0


# Baseline: oldest first (FIFO)
How the shop works today, so we have a number to beat.

**How it works**
1. Load the open jobs from silver.
2. Put the in-house jobs in a queue, oldest `opened_ts` first.
3. Work the queue: 12 tech hours per day, Mon–Fri. A finished car is ready the next morning.
4. Vendor-only jobs (`in_house = false`) are sent to the vendor Monday. Ready = Monday + `vendor_days`.
5. Shortage per class per day = `reservations − cars_ready − cars back so far`, never below 0. Each short car costs one daily rate.
6. Week cost = shop + vendor + lost revenue.

**Assumptions to confirm** (not stated in the brief)
- **A.** A job can span two days. Unfinished work carries over to the next open day.
- **B.** Parts are charged when a job starts, even if it finishes after the week ends.
- **C.** In the baseline, vendor-only jobs are sent Monday, whatever their class.

In [3]:
import pandas as pd
from datetime import date, timedelta

# --- Settings: same values as config/planning.yaml ----------------------------
# Typed in here for now; later we'll read them from the config file instead.
WEEK_START, WEEK_END = date(2026, 10, 12), date(2026, 10, 18)
TECH_HOURS_PER_DAY   = 12
SHOP_OPEN_DAYS       = {"Mon", "Tue", "Wed", "Thu", "Fri"}
LABOR_RATE           = 68
SHOP_READY_DELAY     = timedelta(days=1)      # finished today -> ready tomorrow morning

# --- Step 1: load silver tables into pandas ----------------------------------
# The tables are tiny, and a day-by-day simulation is easier to read in plain Python than in Spark.
silver = lambda t: spark.table(f"{CATALOG}.silver.{t}").toPandas()
work_orders, job_types = silver("work_orders"), silver("job_types")
vehicle_classes, rental_demand = silver("vehicle_classes"), silver("rental_demand")

open_jobs = work_orders[work_orders.status == "open"].merge(job_types, on="job_type")
print(f"Step 1: {len(open_jobs)} open jobs: "
      f"{open_jobs.in_house.sum()} in-house, {(~open_jobs.in_house).sum()} vendor-only")

# --- Step 2: shop queue, oldest first -----------------------------------------
week      = [WEEK_START + timedelta(days=i) for i in range((WEEK_END - WEEK_START).days + 1)]
shop_days = [d for d in week if d.strftime("%a") in SHOP_OPEN_DAYS]

queue = (open_jobs[open_jobs.in_house]
         .sort_values(["opened_ts", "work_order_id"])     # ID breaks ties between equal timestamps
         .reset_index(drop=True))
print(f"Step 2: queue needs {queue.shop_hours.sum()} shop hours; "
      f"the week has {len(shop_days) * TECH_HOURS_PER_DAY} ({len(shop_days)} days x {TECH_HOURS_PER_DAY})")

# --- Step 3: work the queue day by day ----------------------------------------
day_i, hours_left = 0, TECH_HOURS_PER_DAY
starts, finishes, worked = [], [], []

for hours_needed in queue.shop_hours:
    remaining, start, finish = hours_needed, None, None
    while remaining > 1e-9 and day_i < len(shop_days):        # stop when job done or week over
        start = start or shop_days[day_i]
        work = min(remaining, hours_left)
        remaining  -= work
        hours_left -= work
        if remaining <= 1e-9:
            finish = shop_days[day_i]
        if hours_left <= 1e-9:                                  # day used up -> next open day (assumption A)
            day_i, hours_left = day_i + 1, TECH_HOURS_PER_DAY
    starts.append(start); finishes.append(finish); worked.append(hours_needed - remaining)

queue["start_day"]    = starts
queue["finish_day"]   = finishes
queue["hours_worked"] = worked
queue["ready_date"]   = [f + SHOP_READY_DELAY if f else None for f in finishes]
queue["shop_cost"]    = (queue.hours_worked * LABOR_RATE
                         + queue.parts_cost.where(queue.hours_worked > 0, 0))   # assumption B

queue.index = queue.index + 1                                  # queue position 1, 2, 3...
print(f"Step 3: {queue.finish_day.notna().sum()} jobs finished this week, "
      f"{queue.finish_day.isna().sum()} still waiting")
display(queue[["work_order_id", "vehicle_class", "job_type", "opened_ts", "shop_hours",
               "start_day", "finish_day", "ready_date", "shop_cost"]])

# --- Step 4: vendor-only jobs go out Monday (assumption C) --------------------
vendor = open_jobs[~open_jobs.in_house].copy()
vendor["sent_date"]  = WEEK_START
vendor["ready_date"] = [WEEK_START + timedelta(days=int(n)) for n in vendor.vendor_days]
print("Step 4: vendor jobs")
display(vendor[["work_order_id", "vehicle_class", "job_type", "vendor_price", "sent_date", "ready_date"]])

# --- Step 5: shortages per class per day --------------------------------------
cars_back = pd.concat([queue[["vehicle_class", "ready_date"]],
                       vendor[["vehicle_class", "ready_date"]]]).dropna()

demand = (rental_demand[(rental_demand.date >= WEEK_START) & (rental_demand.date <= WEEK_END)]
          .merge(vehicle_classes, on="vehicle_class"))
demand["cars_back"] = [((cars_back.vehicle_class == c) & (cars_back.ready_date <= d)).sum()
                       for c, d in zip(demand.vehicle_class, demand.date)]
demand["short"]        = (demand.reservations - demand.cars_ready - demand.cars_back).clip(lower=0)
demand["lost_revenue"] = demand.short * demand.daily_rate
print("Step 5: cars short per class per day")
display(demand.pivot(index="vehicle_class", columns="date", values="short"))

# --- Step 6: cost of the week -------------------------------------------------
shop_total, vendor_total, lost_total = queue.shop_cost.sum(), vendor.vendor_price.sum(), demand.lost_revenue.sum()
print("Step 6: cost of the week (baseline, oldest first)")
print(f"  Shop cost     ${shop_total:>10,.0f}")
print(f"  Vendor cost   ${vendor_total:>10,.0f}")
print(f"  Lost revenue  ${lost_total:>10,.0f}")
print(f"  TOTAL         ${shop_total + vendor_total + lost_total:>10,.0f}")

Step 1: 54 open jobs: 51 in-house, 3 vendor-only
Step 2: queue needs 89.0 shop hours; the week has 60 (5 days x 12)
Step 3: 35 jobs finished this week, 16 still waiting


,work_order_id,vehicle_class,job_type,opened_ts,shop_hours,start_day,finish_day,ready_date,shop_cost
1,WO-021,economy,tires,2026-10-08 00:00:00,1.50,2026-10-12,2026-10-12,2026-10-13,482.0
2,WO-044,premium suv,battery,2026-10-08 00:00:00,1.50,2026-10-12,2026-10-12,2026-10-13,312.0
3,WO-052,full-size,ac repair,2026-10-08 02:00:00,3.00,2026-10-12,2026-10-12,2026-10-13,624.0
4,WO-016,midsize,oil change,2026-10-08 05:00:00,0.75,2026-10-12,2026-10-12,2026-10-13,96.0
5,WO-014,minivan,oil change,2026-10-08 07:00:00,0.75,2026-10-12,2026-10-12,2026-10-13,96.0
6,WO-030,full-size,brakes,2026-10-08 08:00:00,2.50,2026-10-12,2026-10-12,2026-10-13,430.0
7,WO-048,midsize,ac repair,2026-10-08 15:00:00,3.00,2026-10-12,2026-10-13,2026-10-14,624.0
8,WO-039,full-size,battery,2026-10-08 16:00:00,1.50,2026-10-13,2026-10-13,2026-10-14,312.0
9,WO-025,full-size,tires,2026-10-08 17:00:00,1.50,2026-10-13,2026-10-13,2026-10-14,482.0
10,WO-018,economy,tires,2026-10-08 18:00:00,1.50,2026-10-13,2026-10-13,2026-10-14,482.0


Step 4: vendor jobs


,work_order_id,vehicle_class,job_type,vendor_price,sent_date,ready_date
5,WO-057,full-size,collision repair,2260.0,2026-10-12,2026-10-17
27,WO-055,full-size,transmission,3260.0,2026-10-12,2026-10-19
31,WO-056,economy,transmission,3260.0,2026-10-12,2026-10-19


Step 5: cars short per class per day


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
economy,0,0,0,0,0,0,0
full-size,0,0,0,0,0,0,0
midsize,0,1,1,2,2,3,1
minivan,0,1,1,1,1,1,1
premium suv,0,4,4,3,2,2,2
suv,0,16,13,11,10,7,7


Step 6: cost of the week (baseline, oldest first)
  Shop cost     $    12,685
  Vendor cost   $     8,780
  Lost revenue  $     8,356
  TOTAL         $    29,821


# Priority: short classes first (day by day)
Sets up the **shared day-by-day engine** every rule below uses: it builds the shop order one job at a time, moving forward through the week. Day by day: before every pick, the engine knows the shop clock and which cars are already coming back, and recalculates which jobs would still remove a short car.

**Rule:** jobs that would still remove a short car go first, oldest first; the rest after, oldest first. Vendor-only jobs still go to the vendor, so only the shop order changes.

In [18]:
import os, sys, importlib
from collections import Counter
from pathlib import Path
# use the app's planner + config (works whether the notebook runs from the repo root or notebooks/)
APP_DIR = next(p / "apps" / "hertz-technical-challenge" for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "apps" / "hertz-technical-challenge").exists())
sys.path.insert(0, str(APP_DIR))
from backend import planner
importlib.reload(planner)   # pick up edits to planner.py without restarting the kernel
from backend.config import load_config

cfg = load_config()
inputs = planner.build_inputs(work_orders.to_dict("records"), job_types.to_dict("records"),
                              vehicle_classes.to_dict("records"), rental_demand.to_dict("records"))

# ===== Shared day-by-day planning engine (used by every priority rule below) ==========
EPS = 1e-9
FULL_DAY = cfg.shop.tech_hours_per_day
READY_DELAY = timedelta(days=cfg.shop.ready_delay_days)
week_dates = [cfg.planning_week.start + timedelta(days=i)
              for i in range((cfg.planning_week.end - cfg.planning_week.start).days + 1)]
shop_days = [d for d in week_dates if d.strftime("%a") in cfg.shop.open_days]
gap_before_repairs = {(r.vehicle_class, r.date): r.reservations - r.cars_ready for r in rental_demand.itertuples()}
short_classes = {c for (c, d), g in gap_before_repairs.items() if g > 0 and week_dates[0] <= d <= week_dates[-1]}

def vendor_ready(j):                     # assumption C: vendor jobs go out on the first day of the week
    return week_dates[0] + timedelta(days=j.vendor_days)

def shop_cost(j):
    return j.shop_hours * cfg.shop.labor_rate_per_hour + j.parts_cost

def wo_num(j):
    return int(j.work_order_id.split("-")[1])

class ShopDay:
    """Where the shop is in the week, and which cars are already coming back."""
    def __init__(self, vendor_jobs, split_jobs=True):
        self.day_i, self.left, self.split = 0, FULL_DAY, split_jobs
        self.returns = Counter((j.vehicle_class, vendor_ready(j)) for j in vendor_jobs)

    @property
    def today(self):
        return shop_days[self.day_i] if self.day_i < len(shop_days) else None

    def _work(self, hours, commit):
        day_i, left, finish = self.day_i, self.left, None
        if not self.split and hours > left + EPS and left < FULL_DAY - EPS:
            day_i, left = day_i + 1, FULL_DAY                       # doesn't fit today: start tomorrow
        while hours > EPS and day_i < len(shop_days):
            work = min(hours, left)
            hours, left = hours - work, left - work
            if hours <= EPS:
                finish = shop_days[day_i]
            if left <= EPS:
                day_i, left = day_i + 1, FULL_DAY
        if commit:
            self.day_i, self.left = day_i, left
        return finish

    def ready(self, j):                  # ready date if the shop started this job now
        finish = self._work(j.shop_hours, commit=False)
        return finish + READY_DELAY if finish else None

    def short_on(self, cls, d):          # cars short that day, given the cars already coming back
        back = sum(n for (c, r), n in self.returns.items() if c == cls and r <= d)
        return max(0, gap_before_repairs[(cls, d)] - back)

    def covered(self, j):                # days this job would actually remove a short car
        r = self.ready(j)
        return [d for d in week_dates if r and d >= r and self.short_on(j.vehicle_class, d) > 0]

    def fits_today(self, j):
        return j.shop_hours <= self.left + EPS

    def start(self, j):
        finish = self._work(j.shop_hours, commit=True)
        if finish:
            self.returns[(j.vehicle_class, finish + READY_DELAY)] += 1

    def next_day(self):
        self.day_i, self.left = self.day_i + 1, FULL_DAY

def plan_day_by_day(pick, vendor=(), split_jobs=True, defer=()):
    """Build the shop order one job at a time, moving forward through the week.
    pick(waiting, state) returns the job to start now, or None to end the day.
    vendor: in-house jobs sent to the vendor instead (on the first day).
    defer:  vendor-only jobs held until next week (not sent, not paid this week)."""
    vendor, defer = set(vendor), set(defer)
    to_vendor = [j for j in inputs.jobs.values()
                 if (not j.in_house or j.work_order_id in vendor) and j.work_order_id not in defer]
    state = ShopDay(to_vendor, split_jobs)
    waiting = [j for j in inputs.jobs.values() if j.in_house and j.work_order_id not in vendor]
    order = []
    while waiting:
        if state.today is None:                                       # week over: the rest wait, oldest first
            order += sorted(waiting, key=lambda j: (j.opened_ts, j.work_order_id))
            break
        j = pick(waiting, state)
        if j is None:
            state.next_day()
            continue
        state.start(j)
        order.append(j)
        waiting.remove(j)
    to_vendor.sort(key=lambda j: (j.opened_ts, j.work_order_id))
    return ([planner.PlanItem(j.work_order_id, planner.SHOP) for j in order] +
            [planner.PlanItem(j.work_order_id, planner.VENDOR) for j in to_vendor] +
            [planner.PlanItem(w, planner.HOLD) for w in sorted(defer)])

def by_score(score):
    """Pick rule: start the waiting job with the highest score. Scores are recomputed before every pick."""
    return lambda waiting, state: max(waiting, key=lambda j: score(j, state))

def build(pick, vendor=(), split_jobs=True, defer=()):
    plan = plan_day_by_day(pick, vendor, split_jobs, defer)
    return plan, planner.simulate(plan, inputs, cfg, split_jobs=split_jobs)

# Best rule found (used by several cells): most covered value per shop hour
def score_value_per_hour(j, st):
    value = inputs.daily_rate[j.vehicle_class] * len(st.covered(j))
    return (value / j.shop_hours, -j.opened_ts.timestamp(), -wo_num(j))

# ===== Helpers for the tables below ======================================================
def shortage_grid(w):
    return (pd.DataFrame([(c.vehicle_class, c.date, c.short) for c in w.shortages],
                         columns=["vehicle_class", "date", "short"])
            .pivot(index="vehicle_class", columns="date", values="short"))

def cost_table(names):
    return pd.DataFrame({n: {"Shop": weeks[n].costs.shop, "Vendor": weeks[n].costs.vendor,
                             "Lost revenue": weeks[n].costs.lost_revenue, "TOTAL": weeks[n].costs.total,
                             "Car-days short": sum(c.short for c in weeks[n].shortages)} for n in names})

def cars_back_table(names):
    out = {}
    for n in names:
        ready = [r.ready_date for r in weeks[n].shop_queue if r.ready_date and r.vehicle_class in short_classes]
        ready += [v.ready_date for v in weeks[n].vendor if v.vehicle_class in short_classes]
        out[n] = {d: sum(1 for x in ready if x <= d) for d in week_dates}
    return pd.DataFrame(out).T

def lost_by_class(names):
    return pd.DataFrame({n: pd.Series([c.lost_revenue for c in weeks[n].shortages],
                                      index=[c.vehicle_class for c in weeks[n].shortages]).groupby(level=0).sum()
                         for n in names}).loc[sorted(short_classes)].round(0)

def monday_work(n):
    for r in weeks[n].shop_queue:
        hrs = sum(s.hours for s in r.segments if s.day == shop_days[0])
        if hrs:
            print(f"   #{r.position:<3}{job(r.work_order_id):<40} {hrs:>5} h   ready {r.ready_date or 'next week'}")

job = lambda wo: f"{wo} ({inputs.jobs[wo].vehicle_class}, {inputs.jobs[wo].job_type})"
print("Short classes this week:", sorted(short_classes))

# ===== Rule: short first, oldest first within each group (day by day) ====================
def score_short_first(j, st):
    return (len(st.covered(j)) > 0, -j.opened_ts.timestamp(), -wo_num(j))

plans = {"Oldest first (baseline)": planner.oldest_first_plan(inputs)}
weeks = {"Oldest first (baseline)": planner.simulate(plans["Oldest first (baseline)"], inputs, cfg)}
name = "Short classes first"
plans[name], weeks[name] = build(by_score(score_short_first))

print()
print("1. Cost of the week")
costs = cost_table(["Oldest first (baseline)", name])
display(costs.assign(Change=costs[name] - costs["Oldest first (baseline)"]).round(0))

for n in ["Oldest first (baseline)", name]:
    print()
    print(f"2. Cars short per day: {n}")
    display(shortage_grid(weeks[n]).loc[sorted(short_classes)])

done = {n: {r.work_order_id for r in weeks[n].shop_queue if r.ready_date} for n in weeks}
base, new = done["Oldest first (baseline)"], done[name]
print()
print(f"3. Cars back this week: {len(base)} -> {len(new)}")
print("   Now finished this week:", [job(wo) for wo in sorted(new - base)])
print("   Pushed out of this week:", [job(wo) for wo in sorted(base - new)])

Short classes this week: ['midsize', 'minivan', 'premium suv', 'suv']

1. Cost of the week


,Oldest first (baseline),Short classes first,Change
Shop,12685.0,12410.0,-275.0
Vendor,8780.0,8780.0,0.0
Lost revenue,8356.0,3880.0,-4476.0
TOTAL,29821.0,25070.0,-4751.0
Car-days short,97.0,43.0,-54.0



2. Cars short per day: Oldest first (baseline)


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,1,1,2,2,3,1
minivan,0,1,1,1,1,1,1
premium suv,0,4,4,3,2,2,2
suv,0,16,13,11,10,7,7



2. Cars short per day: Short classes first


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,0,0,0,0,0,0
minivan,0,1,1,1,0,0,0
premium suv,0,4,2,2,1,0,0
suv,0,12,10,4,3,1,1



3. Cars back this week: 35 -> 36
   Now finished this week: ['WO-003 (midsize, oil change)', 'WO-006 (premium suv, oil change)', 'WO-013 (premium suv, oil change)', 'WO-017 (suv, tires)', 'WO-020 (midsize, tires)', 'WO-026 (suv, tires)', 'WO-028 (minivan, tires)', 'WO-029 (suv, brakes)', 'WO-035 (suv, brakes)', 'WO-036 (suv, brakes)', 'WO-050 (midsize, ac repair)', 'WO-053 (suv, ac repair)']
   Pushed out of this week: ['WO-001 (economy, oil change)', 'WO-009 (full-size, oil change)', 'WO-018 (economy, tires)', 'WO-022 (full-size, tires)', 'WO-025 (full-size, tires)', 'WO-034 (economy, brakes)', 'WO-038 (economy, battery)', 'WO-043 (economy, battery)', 'WO-046 (economy, ac repair)', 'WO-049 (economy, ac repair)', 'WO-051 (full-size, ac repair)']


# Short first + shortest job first (day by day)
Day by day: before every pick, the engine knows the shop clock and which cars are already coming back, and recalculates which jobs would still remove a short car.

**Rule:** jobs that would still remove a short car first, **shortest job first** (ties: oldest). Quick jobs get more cars back sooner.

In [19]:
def score_shortest(j, st):
    c = len(st.covered(j)) > 0
    return (c, -j.shop_hours if c else 0, -j.opened_ts.timestamp(), -wo_num(j))

name = "Short first + shortest job"
plans[name], weeks[name] = build(by_score(score_shortest))

print("1. Cost of the week")
costs = cost_table(["Oldest first (baseline)", "Short classes first", name])
display(costs.assign(**{"vs baseline": costs[name] - costs["Oldest first (baseline)"],
                        "vs short first": costs[name] - costs["Short classes first"]}).round(0))

print()
print(f"2. Cars short per day: {name}")
display(shortage_grid(weeks[name]).loc[sorted(short_classes)])

print()
print("3. Short-class cars back in service by each morning")
display(cars_back_table(["Oldest first (baseline)", "Short classes first", name]))

prev = {r.work_order_id: r.ready_date for r in weeks["Short classes first"].shop_queue}
now = {r.work_order_id: r.ready_date for r in weeks[name].shop_queue}
earlier = [f"{job(wo)}: {prev[wo] or 'next week'} -> {now[wo]}" for wo in now
           if now[wo] and (prev[wo] is None or now[wo] < prev[wo])]
later = [f"{job(wo)}: {prev[wo]} -> {now[wo] or 'next week'}" for wo in now
         if prev[wo] and (now[wo] is None or now[wo] > prev[wo])]
print()
print(f"4. Ready earlier than with short classes first ({len(earlier)}):")
print("\n".join("   " + x for x in earlier))
print(f"   Ready later ({len(later)}):")
print("\n".join("   " + x for x in later))

1. Cost of the week


,Oldest first (baseline),Short classes first,Short first + shortest job,vs baseline,vs short first
Shop,12685.0,12410.0,12630.0,-55.0,220.0
Vendor,8780.0,8780.0,8780.0,0.0,0.0
Lost revenue,8356.0,3880.0,2778.0,-5578.0,-1102.0
TOTAL,29821.0,25070.0,24188.0,-5633.0,-882.0
Car-days short,97.0,43.0,32.0,-65.0,-11.0



2. Cars short per day: Short first + shortest job


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,0,0,0,0,0,0
minivan,0,1,0,0,0,0,0
premium suv,0,2,1,1,0,0,0
suv,0,11,8,4,2,1,1



3. Short-class cars back in service by each morning


,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
Oldest first (baseline),0,3,8,12,16,20,20
Short classes first,0,9,17,24,29,32,32
Short first + shortest job,0,14,22,26,31,32,32



4. Ready earlier than with short classes first (14):
   WO-011 (midsize, oil change): 2026-10-14 -> 2026-10-13
   WO-005 (midsize, oil change): 2026-10-14 -> 2026-10-13
   WO-010 (suv, oil change): 2026-10-15 -> 2026-10-13
   WO-003 (midsize, oil change): 2026-10-15 -> 2026-10-13
   WO-006 (premium suv, oil change): 2026-10-16 -> 2026-10-13
   WO-013 (premium suv, oil change): 2026-10-17 -> 2026-10-13
   WO-042 (suv, battery): 2026-10-14 -> 2026-10-13
   WO-026 (suv, tires): 2026-10-15 -> 2026-10-14
   WO-017 (suv, tires): 2026-10-15 -> 2026-10-14
   WO-020 (midsize, tires): 2026-10-16 -> 2026-10-14
   WO-028 (minivan, tires): 2026-10-16 -> 2026-10-14
   WO-036 (suv, brakes): 2026-10-16 -> 2026-10-15
   WO-033 (midsize, brakes): next week -> 2026-10-16
   WO-035 (suv, brakes): 2026-10-17 -> 2026-10-16
   Ready later (5):
   WO-032 (suv, brakes): 2026-10-13 -> 2026-10-15
   WO-048 (midsize, ac repair): 2026-10-13 -> 2026-10-16
   WO-054 (premium suv, ac repair): 2026-10-14 -> 2026-10-1

# Short first + most profitable (day by day)
Day by day: before every pick, the engine knows the shop clock and which cars are already coming back, and recalculates which jobs would still remove a short car.

**Rule:** jobs that would still remove a short car first, **highest daily rental rate first**, then shortest job, then oldest.

In [20]:
def score_most_profitable(j, st):
    c = len(st.covered(j)) > 0
    return (c, inputs.daily_rate[j.vehicle_class] if c else 0, -j.shop_hours if c else 0,
            -j.opened_ts.timestamp(), -wo_num(j))

name = "Short first + most profitable"
plans[name], weeks[name] = build(by_score(score_most_profitable))
compare = ["Oldest first (baseline)", "Short classes first", "Short first + shortest job", name]

print("1. Cost of the week")
costs = cost_table(compare)
display(costs.assign(**{"vs baseline": costs[name] - costs["Oldest first (baseline)"],
                        "vs shortest job": costs[name] - costs["Short first + shortest job"]}).round(0))

print()
print(f"2. Cars short per day: {name}")
display(shortage_grid(weeks[name]).loc[sorted(short_classes)])

print()
print("3. Lost revenue by class")
display(lost_by_class(compare))

print()
print("4. Short-class cars back in service by each morning")
display(cars_back_table(compare))

print()
print(f"5. Monday's 12 hours under '{name}'")
monday_work(name)

1. Cost of the week


,Oldest first (baseline),Short classes first,Short first + shortest job,Short first + most profitable,vs baseline,vs shortest job
Shop,12685.0,12410.0,12630.0,12630.0,-55.0,0.0
Vendor,8780.0,8780.0,8780.0,8780.0,0.0,0.0
Lost revenue,8356.0,3880.0,2778.0,2628.0,-5728.0,-150.0
TOTAL,29821.0,25070.0,24188.0,24038.0,-5783.0,-150.0
Car-days short,97.0,43.0,32.0,36.0,-61.0,4.0



2. Cars short per day: Short first + most profitable


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,2,4,6,0,0,0
minivan,0,0,0,0,0,0,0
premium suv,0,0,0,0,0,0,0
suv,0,13,6,2,1,1,1



3. Lost revenue by class


,Oldest first (baseline),Short classes first,Short first + shortest job,Short first + most profitable
midsize,550.0,0.0,0.0,660.0
minivan,552.0,276.0,92.0,0.0
premium suv,2006.0,1062.0,472.0,0.0
suv,5248.0,2542.0,2214.0,1968.0



4. Short-class cars back in service by each morning


,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
Oldest first (baseline),0,3,8,12,16,20,20
Short classes first,0,9,17,24,29,32,32
Short first + shortest job,0,14,22,26,31,32,32
Short first + most profitable,0,10,17,21,31,32,32



5. Monday's 12 hours under 'Short first + most profitable'
   #1  WO-006 (premium suv, oil change)          0.75 h   ready 2026-10-13
   #2  WO-013 (premium suv, oil change)          0.75 h   ready 2026-10-13
   #3  WO-044 (premium suv, battery)              1.5 h   ready 2026-10-13
   #4  WO-024 (premium suv, tires)                1.5 h   ready 2026-10-13
   #5  WO-054 (premium suv, ac repair)            3.0 h   ready 2026-10-13
   #6  WO-014 (minivan, oil change)              0.75 h   ready 2026-10-13
   #7  WO-028 (minivan, tires)                    1.5 h   ready 2026-10-13
   #8  WO-007 (suv, oil change)                  0.75 h   ready 2026-10-13
   #9  WO-002 (suv, oil change)                  0.75 h   ready 2026-10-13
   #10 WO-012 (suv, oil change)                  0.75 h   ready 2026-10-13


# Most profitable + best fit into each day's 12 hours (day by day)
Day by day: before every pick, the engine knows the shop clock and which cars are already coming back, and recalculates which jobs would still remove a short car.

**Rule:** whole jobs only, so no car is left half-done overnight. Take the next job in most-profitable order if it fits in today's remaining hours; otherwise fill the gap with the job that fits **best** (fewest hours left over; jobs that still remove a short car first). When nothing fits, the day ends. Simulated with `split_jobs=False`.

In [21]:
def best_fit_pick(waiting, st):
    """Whole jobs only. Take the next job in most-profitable order if it fits today;
    otherwise fill the gap with the job that fits best (short-class jobs first); if nothing fits, end the day."""
    prio = sorted(waiting, key=lambda j: score_most_profitable(j, st), reverse=True)
    fits = [j for j in prio if st.fits_today(j)]
    if not fits:
        return None
    if st.fits_today(prio[0]):
        return prio[0]
    pool = [j for j in fits if st.covered(j)] or fits
    return min(pool, key=lambda j: (st.left - j.shop_hours, prio.index(j)))

name = "Most profitable + best fit"
plans[name], weeks[name] = build(best_fit_pick, split_jobs=False)
compare = ["Oldest first (baseline)", "Short classes first", "Short first + shortest job",
           "Short first + most profitable", name]

print("1. Cost of the week")
costs = cost_table(compare)
display(costs.assign(**{"vs baseline": costs[name] - costs["Oldest first (baseline)"],
                        "vs most profitable": costs[name] - costs["Short first + most profitable"]}).round(0))

print()
print("2. Each day's hours (whole jobs only)")
for d in shop_days:
    today = [(r, sum(s.hours for s in r.segments if s.day == d)) for r in weeks[name].shop_queue]
    today = [(r, h) for r, h in today if h]
    used = sum(h for _, h in today)
    print()
    print(f"   {d:%a %m/%d}: {used:g} of {FULL_DAY:g} h used, {FULL_DAY - used:g} h idle")
    for r, h in today:
        print(f"      {job(r.work_order_id):<40} {h:>5} h")

print()
print(f"3. Cars short per day: {name}")
display(shortage_grid(weeks[name]).loc[sorted(short_classes)])

print()
print("4. Lost revenue by class")
display(lost_by_class(compare))

print()
print("5. Short-class cars back in service by each morning")
display(cars_back_table(compare))

1. Cost of the week


,Oldest first (baseline),Short classes first,Short first + shortest job,Short first + most profitable,Most profitable + best fit,vs baseline,vs most profitable
Shop,12685.0,12410.0,12630.0,12630.0,12096.0,-589.0,-534.0
Vendor,8780.0,8780.0,8780.0,8780.0,8780.0,0.0,0.0
Lost revenue,8356.0,3880.0,2778.0,2628.0,2573.0,-5783.0,-55.0
TOTAL,29821.0,25070.0,24188.0,24038.0,23449.0,-6372.0,-589.0
Car-days short,97.0,43.0,32.0,36.0,35.0,-62.0,-1.0



2. Each day's hours (whole jobs only)

   Mon 10/12: 12 of 12 h used, 0 h idle
      WO-006 (premium suv, oil change)          0.75 h
      WO-013 (premium suv, oil change)          0.75 h
      WO-044 (premium suv, battery)              1.5 h
      WO-024 (premium suv, tires)                1.5 h
      WO-054 (premium suv, ac repair)            3.0 h
      WO-014 (minivan, oil change)              0.75 h
      WO-028 (minivan, tires)                    1.5 h
      WO-007 (suv, oil change)                  0.75 h
      WO-002 (suv, oil change)                  0.75 h
      WO-012 (suv, oil change)                  0.75 h

   Tue 10/13: 11.75 of 12 h used, 0.25 h idle
      WO-010 (suv, oil change)                  0.75 h
      WO-042 (suv, battery)                      1.5 h
      WO-040 (suv, battery)                      1.5 h
      WO-026 (suv, tires)                        1.5 h
      WO-017 (suv, tires)                        1.5 h
      WO-032 (suv, brakes)                      

date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,2,4,5,0,0,0
minivan,0,0,0,0,0,0,0
premium suv,0,0,0,0,0,0,0
suv,0,13,6,2,1,1,1



4. Lost revenue by class


,Oldest first (baseline),Short classes first,Short first + shortest job,Short first + most profitable,Most profitable + best fit
midsize,550.0,0.0,0.0,660.0,605.0
minivan,552.0,276.0,92.0,0.0,0.0
premium suv,2006.0,1062.0,472.0,0.0,0.0
suv,5248.0,2542.0,2214.0,1968.0,1968.0



5. Short-class cars back in service by each morning


,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
Oldest first (baseline),0,3,8,12,16,20,20
Short classes first,0,9,17,24,29,32,32
Short first + shortest job,0,14,22,26,31,32,32
Short first + most profitable,0,10,17,21,31,32,32
Most profitable + best fit,0,10,17,22,30,32,32


# Most profitable + send still-short cars to the vendor (day by day)
Start from the most-profitable day-by-day plan. While a short class is still short: send out its job that would come back **latest** (cheaper vendor price breaks ties), **re-plan the shop day by day** with that car coming back from the vendor, and keep the send only if it reduces cars short. Stops when nothing helps or the budget (after the vendor-only jobs) runs out. Each send shows its effect on the week's total.

In [22]:
def send_still_short_to_vendor(pick):
    """Re-plan the shop (day by day) after every send, so freed hours and vendor returns are used."""
    vendor = set()
    plan, week = build(pick)
    sends, tried = [], set()
    while True:
        still_short = {c.vehicle_class for c in week.shortages if c.short > 0} & short_classes
        cands = [r for r in week.shop_queue
                 if r.vehicle_class in still_short and r.work_order_id not in tried
                 and inputs.jobs[r.work_order_id].vendor_price <= week.costs.budget_left]
        if not cands:
            break
        # latest back first: not this week, then latest ready date; cheaper vendor price breaks ties
        cands.sort(key=lambda r: (r.ready_date is not None,
                                  -(r.ready_date.toordinal() if r.ready_date else 0),
                                  inputs.jobs[r.work_order_id].vendor_price))
        r = cands[0]
        tried.add(r.work_order_id)
        new_plan, new_week = build(pick, vendor | {r.work_order_id})
        if new_week.costs.lost_revenue >= week.costs.lost_revenue - EPS:
            continue                                                  # didn't reduce any shortage: skip
        sends.append({"order": r.work_order_id, "class": r.vehicle_class, "job": r.job_type,
                      "shop ready": r.ready_date or "next week", "vendor ready": vendor_ready(inputs.jobs[r.work_order_id]),
                      "vendor price": inputs.jobs[r.work_order_id].vendor_price,
                      "lost revenue change": new_week.costs.lost_revenue - week.costs.lost_revenue,
                      "week cost change": new_week.costs.total - week.costs.total,
                      "budget left": new_week.costs.budget_left})
        vendor.add(r.work_order_id)
        plan, week = new_plan, new_week
    return plan, week, sends

name = "Most profitable + vendor for still-short"
plans[name], weeks[name], sends = send_still_short_to_vendor(by_score(score_most_profitable))
compare = ["Oldest first (baseline)", "Short classes first", "Short first + most profitable", name]

print("1. Cost of the week")
costs = cost_table(compare)
costs.loc["Budget left"] = [weeks[n].costs.budget_left for n in compare]
display(costs.assign(**{"vs baseline": costs[name] - costs["Oldest first (baseline)"],
                        "vs most profitable": costs[name] - costs["Short first + most profitable"]}).round(0))

print()
print(f"2. Sent to the vendor ({len(sends)} jobs)")
display(pd.DataFrame(sends).round(0) if sends else "Nothing sent: no send reduced cars short within the budget.")

print()
print(f"3. Cars short per day: {name}")
display(shortage_grid(weeks[name]).loc[sorted(short_classes)])

print()
print("4. Lost revenue by class")
display(lost_by_class(compare))

1. Cost of the week


,Oldest first (baseline),Short classes first,Short first + most profitable,Most profitable + vendor for still-short,vs baseline,vs most profitable
Shop,12685.0,12410.0,12630.0,12625.0,-60.0,-5.0
Vendor,8780.0,8780.0,8780.0,13940.0,5160.0,5160.0
Lost revenue,8356.0,3880.0,2628.0,1640.0,-6716.0,-988.0
TOTAL,29821.0,25070.0,24038.0,28205.0,-1616.0,4167.0
Car-days short,97.0,43.0,36.0,20.0,-77.0,-16.0
Budget left,5220.0,5220.0,5220.0,60.0,-5160.0,-5160.0



2. Sent to the vendor (8 jobs)


,order,class,job,shop ready,vendor ready,vendor price,lost revenue change,week cost change,budget left
0,WO-050,midsize,ac repair,next week,2026-10-14,840.0,-110.0,730.0,4380.0
1,WO-048,midsize,ac repair,2026-10-17,2026-10-14,840.0,-110.0,850.0,3540.0
2,WO-033,midsize,brakes,next week,2026-10-13,540.0,-165.0,375.0,3000.0
3,WO-020,midsize,tires,next week,2026-10-13,580.0,-165.0,510.0,2420.0
4,WO-041,midsize,battery,next week,2026-10-13,400.0,-55.0,-35.0,2020.0
5,WO-027,midsize,tires,next week,2026-10-13,580.0,-55.0,525.0,1440.0
6,WO-053,suv,ac repair,2026-10-16,2026-10-14,840.0,-164.0,676.0,600.0
7,WO-029,suv,brakes,2026-10-15,2026-10-13,540.0,-164.0,536.0,60.0



3. Cars short per day: Most profitable + vendor for still-short


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,0,0,0,0,0,0
minivan,0,0,0,0,0,0,0
premium suv,0,0,0,0,0,0,0
suv,0,12,4,1,1,1,1



4. Lost revenue by class


,Oldest first (baseline),Short classes first,Short first + most profitable,Most profitable + vendor for still-short
midsize,550.0,0.0,660.0,0.0
minivan,552.0,276.0,0.0,0.0
premium suv,2006.0,1062.0,0.0,0.0
suv,5248.0,2542.0,1968.0,1640.0


# Net profit per car, then shop or vendor per car (day by day)
Day by day: before every pick, the engine knows the shop clock and which cars are already coming back, and recalculates which jobs would still remove a short car.

**Rule:** score = what the car would **actually** earn this week if started now (daily rate × the short days it would cover) − its shop repair cost. Highest first; jobs that cover nothing go last, oldest first.

**Shop or vendor:** go through short-class cars in shop order; send one out if the days the vendor saves (while its class is still short) × daily rate beat the vendor's extra price, and the budget allows. The shop is re-planned day by day after every send.

In [23]:
def score_net_profit(j, st):
    covered = st.covered(j)
    if not covered:
        return (False, 0, -j.opened_ts.timestamp(), -wo_num(j))
    rental = inputs.daily_rate[j.vehicle_class] * len(covered)      # what it would actually earn this week
    return (True, rental - shop_cost(j), -j.opened_ts.timestamp(), -wo_num(j))

net_profit_pick = by_score(score_net_profit)
plans["Net-profit order"], weeks["Net-profit order"] = build(net_profit_pick)

def choose_shop_or_vendor(pick):
    """Go through short-class cars in shop order; send one out if the days the vendor saves
    (while its class is still short) x daily rate beat the vendor's extra price. Re-plan after each send."""
    vendor = set()
    plan, week = build(pick)
    decisions = []
    for wo in [r.work_order_id for r in week.shop_queue if r.vehicle_class in short_classes]:
        row = next((r for r in week.shop_queue if r.work_order_id == wo), None)
        if row is None:
            continue
        j = inputs.jobs[wo]
        shop_ready = row.ready_date or (week_dates[-1] + timedelta(days=1))
        still_short = {c.date for c in week.shortages if c.vehicle_class == j.vehicle_class and c.short > 0}
        days_saved = [d for d in week_dates if vendor_ready(j) <= d < shop_ready and d in still_short]
        gain = inputs.daily_rate[j.vehicle_class] * len(days_saved)
        extra = j.vendor_price - shop_cost(j)
        send = gain > extra and j.vendor_price <= week.costs.budget_left
        result = {"order": wo, "class": j.vehicle_class, "job": j.job_type,
                  "shop cost": shop_cost(j), "vendor price": j.vendor_price,
                  "shop ready": row.ready_date or "next week", "vendor ready": vendor_ready(j),
                  "rental gained": gain, "vendor extra": extra, "decision": "VENDOR" if send else "shop"}
        if send:
            new_plan, new_week = build(pick, vendor | {wo})
            result["actual week change"] = new_week.costs.total - week.costs.total
            vendor.add(wo)
            plan, week = new_plan, new_week
        decisions.append(result)
    return plan, week, decisions

name = "Net profit + shop/vendor per car"
plans[name], weeks[name], car_decisions = choose_shop_or_vendor(net_profit_pick)
compare = ["Oldest first (baseline)", "Short first + most profitable", "Most profitable + vendor for still-short",
           "Net-profit order", name]

print("1. Cost of the week")
costs = cost_table(compare)
costs.loc["Budget left"] = [weeks[n].costs.budget_left for n in compare]
display(costs.assign(**{"vs baseline": costs[name] - costs["Oldest first (baseline)"],
                        "vs net-profit order": costs[name] - costs["Net-profit order"]}).round(0))

print()
print("2. Every short-class car, in shop order, and the shop/vendor decision")
display(pd.DataFrame(car_decisions).round(0))

print()
print(f"3. Cars short per day: {name}")
display(shortage_grid(weeks[name]).loc[sorted(short_classes)])

print()
print("4. Short-class cars back in service by each morning")
display(cars_back_table(compare))

1. Cost of the week


,Oldest first (baseline),Short first + most profitable,Most profitable + vendor for still-short,Net-profit order,Net profit + shop/vendor per car,vs baseline,vs net-profit order
Shop,12685.0,12630.0,12625.0,12630.0,12835.0,150.0,205.0
Vendor,8780.0,8780.0,13940.0,8780.0,11020.0,2240.0,2240.0
Lost revenue,8356.0,2628.0,1640.0,2378.0,1558.0,-6798.0,-820.0
TOTAL,29821.0,24038.0,28205.0,23788.0,25413.0,-4408.0,1625.0
Car-days short,97.0,36.0,20.0,28.0,18.0,-79.0,-10.0
Budget left,5220.0,5220.0,60.0,5220.0,2980.0,-2240.0,-2240.0



2. Every short-class car, in shop order, and the shop/vendor decision


,order,class,job,shop cost,vendor price,shop ready,vendor ready,rental gained,vendor extra,decision,actual week change
0,WO-006,premium suv,oil change,96.0,195.0,2026-10-13,2026-10-13,0.0,99.0,shop,NaN
1,WO-013,premium suv,oil change,96.0,195.0,2026-10-13,2026-10-13,0.0,99.0,shop,NaN
2,WO-014,minivan,oil change,96.0,195.0,2026-10-13,2026-10-13,0.0,99.0,shop,NaN
3,WO-044,premium suv,battery,312.0,400.0,2026-10-13,2026-10-13,0.0,88.0,shop,NaN
4,WO-007,suv,oil change,96.0,195.0,2026-10-13,2026-10-13,0.0,99.0,shop,NaN
5,WO-002,suv,oil change,96.0,195.0,2026-10-13,2026-10-13,0.0,99.0,shop,NaN
6,WO-012,suv,oil change,96.0,195.0,2026-10-13,2026-10-13,0.0,99.0,shop,NaN
7,WO-010,suv,oil change,96.0,195.0,2026-10-13,2026-10-13,0.0,99.0,shop,NaN
8,WO-016,midsize,oil change,96.0,195.0,2026-10-13,2026-10-13,0.0,99.0,shop,NaN
9,WO-004,midsize,oil change,96.0,195.0,2026-10-13,2026-10-13,0.0,99.0,shop,NaN



3. Cars short per day: Net profit + shop/vendor per car


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,0,0,0,0,0,0
minivan,0,1,0,0,0,0,0
premium suv,0,1,1,0,0,0,0
suv,0,7,3,2,1,1,1



4. Short-class cars back in service by each morning


,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
Oldest first (baseline),0,3,8,12,16,20,20
Short first + most profitable,0,10,17,21,31,32,32
Most profitable + vendor for still-short,0,15,25,32,32,32,32
Net-profit order,0,13,20,25,31,32,32
Net profit + shop/vendor per car,0,17,24,29,32,32,32


# Vendor sends as a 0/1 knapsack (day by day)
Start from the net-profit day-by-day plan. Each in-house short-class job is an item: **weight = vendor price**, **capacity = vendor budget left**, **value = what sending it saves**. An exact 0/1 knapsack (dynamic programming over whole dollars) picks the best set within budget.

Two kinds of value: **estimated** (per-car rental gained − vendor extra) and **simulated** (send it alone, re-plan the shop day by day, compare the week's cost). The chosen set is then sent **together** to check knapsack's assumption that values add up.

In [24]:
def knapsack(items, capacity):
    """Exact 0/1 knapsack. items: list of (key, weight in whole dollars, value). Returns chosen keys."""
    items = [it for it in items if it[2] > 0 and it[1] <= capacity]     # only items that can help and fit
    best = [0.0] * (capacity + 1)
    take = [[False] * (capacity + 1) for _ in items]
    for i, (_, w, v) in enumerate(items):
        for c in range(capacity, w - 1, -1):                            # backwards: each item used at most once
            if best[c - w] + v > best[c]:
                best[c], take[i][c] = best[c - w] + v, True
    chosen, c = [], capacity
    for i in range(len(items) - 1, -1, -1):                            # walk back to recover the set
        if take[i][c]:
            chosen.append(items[i][0])
            c -= items[i][1]
    return chosen[::-1]

base_plan, base = build(net_profit_pick)
capacity = int(base.costs.budget_left)
candidates = [r for r in base.shop_queue if r.vehicle_class in short_classes]

def estimated_value(r):
    j = inputs.jobs[r.work_order_id]
    shop_ready = r.ready_date or (week_dates[-1] + timedelta(days=1))
    still_short = {c.date for c in base.shortages if c.vehicle_class == j.vehicle_class and c.short > 0}
    gain = inputs.daily_rate[j.vehicle_class] * sum(1 for d in week_dates if vendor_ready(j) <= d < shop_ready and d in still_short)
    return gain - (j.vendor_price - shop_cost(j))

def simulated_value(r):                     # send it alone, re-plan the shop day by day, compare week cost
    return base.costs.total - build(net_profit_pick, {r.work_order_id})[1].costs.total

values = pd.DataFrame([{"order": r.work_order_id, "class": r.vehicle_class, "job": r.job_type,
                        "vendor price": inputs.jobs[r.work_order_id].vendor_price,
                        "estimated value": estimated_value(r), "simulated value": simulated_value(r)}
                       for r in candidates])

print(f"Starting plan: net-profit order, week cost ${base.costs.total:,.0f}, vendor budget left ${capacity:,}")
print()
print("1. Every candidate's value (positive = sending it saves money)")
display(values.sort_values("simulated value", ascending=False).round(0))

results = {}
for kind in ["estimated value", "simulated value"]:
    items = [(row.order, int(row["vendor price"]), row[kind]) for _, row in values.iterrows()]
    chosen = knapsack(items, capacity)
    together = build(net_profit_pick, set(chosen))[1]
    v = values.set_index("order")
    results[kind] = {"jobs sent": ", ".join(chosen) or "none", "count": len(chosen),
                     "vendor spend": v.loc[chosen, "vendor price"].sum() if chosen else 0,
                     "predicted saving": v.loc[chosen, kind].sum() if chosen else 0.0,
                     "actual saving (sent together)": base.costs.total - together.costs.total,
                     "week cost": together.costs.total, "lost revenue": together.costs.lost_revenue,
                     "budget left": together.costs.budget_left}

print()
print("2. Knapsack choice with each kind of value, and what happens when the chosen jobs are sent together")
display(pd.DataFrame(results).T.round(0))

Starting plan: net-profit order, week cost $23,788, vendor budget left $5,220

1. Every candidate's value (positive = sending it saves money)


,order,class,job,vendor price,estimated value,simulated value
0,WO-006,premium suv,oil change,195.0,-99.0,14.0
1,WO-013,premium suv,oil change,195.0,-99.0,14.0
2,WO-014,minivan,oil change,195.0,-99.0,14.0
4,WO-007,suv,oil change,195.0,-99.0,14.0
5,WO-002,suv,oil change,195.0,-99.0,14.0
7,WO-010,suv,oil change,195.0,-99.0,14.0
6,WO-012,suv,oil change,195.0,-99.0,14.0
8,WO-016,midsize,oil change,195.0,-99.0,14.0
9,WO-004,midsize,oil change,195.0,-99.0,14.0
14,WO-003,midsize,oil change,195.0,-99.0,14.0



2. Knapsack choice with each kind of value, and what happens when the chosen jobs are sent together


,jobs sent,count,vendor spend,predicted saving,actual saving (sent together),week cost,lost revenue,budget left
estimated value,"WO-036, WO-035, WO-026, WO-017",4,2240.0,322.0,-1625.0,25413.0,1558.0,2980.0
simulated value,"WO-006, WO-013, WO-014, WO-007, WO-002, WO-012...",12,2340.0,168.0,-2301.0,26089.0,1394.0,2880.0


# Vendor options: best savings vs fewest short cars (day by day)
All start from the net-profit day-by-day plan and send in-house short-class jobs one at a time, **re-planning the shop day by day after every send**:
- **A. Best savings**: the send that lowers the week's total most; stop when none does.
- **B. Fewest short, within budget**: the send that removes the most lost revenue per vendor dollar; stop when the budget runs out or nothing helps.
- **C. Fewest short, no budget limit**: like B with no budget, to show the lowest shortage possible and its cost.

**The floor**: a class can't get below *reservations − cars ready − every car of that class in repair*.

In [25]:
def vendor_option(pick, goal, budget=None, defer=()):
    """Send in-house short-class jobs to the vendor one at a time, re-planning the shop day by day after each.
    goal='savings': pick the send that lowers the week's total most; stop when none does.
    goal='short':   pick the send that removes the most lost revenue per vendor dollar; stop when none does
                    or the budget runs out (budget=None -> the plan's own budget; float('inf') -> no limit)."""
    vendor, spent = set(), 0.0
    plan, week = build(pick, defer=defer)
    sends = []
    while True:
        left = week.costs.budget_left if budget is None else budget - spent
        best = None
        for r in week.shop_queue:
            j = inputs.jobs[r.work_order_id]
            if r.vehicle_class not in short_classes or j.vendor_price > left:
                continue
            trial_plan, w = build(pick, vendor | {r.work_order_id}, defer=defer)
            if goal == "savings":
                score = week.costs.total - w.costs.total
            else:
                cut = week.costs.lost_revenue - w.costs.lost_revenue
                score = cut / j.vendor_price if cut > EPS else 0
            if score > EPS and (best is None or score > best[0]):
                best = (score, r, trial_plan, w)
        if best is None:
            break
        _, r, plan, new_week = best
        sends.append({"order": r.work_order_id, "class": r.vehicle_class, "job": r.job_type,
                      "vendor price": inputs.jobs[r.work_order_id].vendor_price,
                      "lost revenue change": new_week.costs.lost_revenue - week.costs.lost_revenue,
                      "week cost change": new_week.costs.total - week.costs.total})
        vendor.add(r.work_order_id)
        spent += inputs.jobs[r.work_order_id].vendor_price
        week = new_week
    return plan, week, sends

start_plan, start = build(net_profit_pick)
options = {
    "Start (net-profit order)": (start_plan, start, []),
    "A. Best savings": vendor_option(net_profit_pick, "savings"),
    "B. Fewest short, within budget": vendor_option(net_profit_pick, "short"),
    "C. Fewest short, no budget limit": vendor_option(net_profit_pick, "short", budget=float("inf")),
}

# --- The floor: shortage that no repair plan can remove --------------------------
in_repair = pd.Series([j.vehicle_class for j in inputs.jobs.values()]).value_counts()
floor = (rental_demand.assign(floor=lambda d: (d.reservations - d.cars_ready - d.vehicle_class.map(in_repair).fillna(0)).clip(lower=0))
         .pivot(index="vehicle_class", columns="date", values="floor").loc[sorted(short_classes)])
print("0. The floor: cars short even if every car in repair were back on the first day")
display(floor)

print()
print("1. The options")
display(pd.DataFrame({n: {"Jobs sent to vendor (extra)": len(s), "Shop": w.costs.shop, "Vendor": w.costs.vendor,
                          "Lost revenue": w.costs.lost_revenue, "TOTAL": w.costs.total,
                          "Budget left": w.costs.budget_left,
                          "Car-days short": sum(c.short for c in w.shortages)}
                      for n, (_, w, s) in options.items()}).round(0))

for n, (_, w, s) in options.items():
    if n.startswith("Start"):
        continue
    print()
    print(f"2. {n}: sent {len(s)} jobs")
    if s:
        display(pd.DataFrame(s).round(0))
    print("   Cars short per day:")
    display(shortage_grid(w).loc[sorted(short_classes)])

0. The floor: cars short even if every car in repair were back on the first day


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,0,0,0,0,0,0
minivan,0,0,0,0,0,0,0
premium suv,0,0,0,0,0,0,0
suv,0,1,1,1,1,1,1



1. The options


,Start (net-profit order),A. Best savings,"B. Fewest short, within budget","C. Fewest short, no budget limit"
Jobs sent to vendor (extra),0.0,1.0,10.0,10.0
Shop,12630.0,12585.0,12890.0,12890.0
Vendor,8780.0,8975.0,13610.0,13610.0
Lost revenue,2378.0,2214.0,774.0,774.0
TOTAL,23788.0,23774.0,27274.0,27274.0
Budget left,5220.0,5025.0,390.0,390.0
Car-days short,28.0,26.0,9.0,9.0



2. A. Best savings: sent 1 jobs


,order,class,job,vendor price,lost revenue change,week cost change
0,WO-006,premium suv,oil change,195.0,-164.0,-14.0


... (6 outputs left out)


2. C. Fewest short, no budget limit: sent 10 jobs


,order,class,job,vendor price,lost revenue change,week cost change
0,WO-006,premium suv,oil change,195.0,-164.0,-14.0
1,WO-032,suv,brakes,540.0,-164.0,706.0
2,WO-037,suv,brakes,540.0,-164.0,706.0
3,WO-013,premium suv,oil change,195.0,-164.0,-14.0
4,WO-029,suv,brakes,540.0,-200.0,385.0
5,WO-036,suv,brakes,540.0,-246.0,499.0
6,WO-028,minivan,tires,580.0,-92.0,108.0
7,WO-035,suv,brakes,540.0,-164.0,536.0
8,WO-026,suv,tires,580.0,-82.0,118.0
9,WO-017,suv,tires,580.0,-164.0,456.0


   Cars short per day:


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,0,0,0,0,0,0
minivan,0,0,0,0,0,0,0
premium suv,0,1,0,0,0,0,0
suv,0,3,1,1,1,1,1


# Defer spare-class vendor jobs; spend the budget to cover the short classes
Vendor-only jobs in classes that **aren't short** add nothing this week, so hold them until next week and use the whole weekly budget on the short classes:

1. **Defer** vendor-only jobs in classes that aren't short (sent next week instead).
2. **Cars needed by the first short morning**, per class: reservations − cars ready, capped at the cars in repair (the floor can't be beaten).
3. **Monday's 12 shop hours as a 0/1 knapsack** over those cars: weight = shop hours, capacity = 12 hours, value = the vendor price the shop avoids (plus a day of rental if the vendor can't make the first morning, e.g. 2-day AC jobs). Chosen jobs are done in the shop Monday; **the rest go to the vendor Monday**. Here each job's value is its own vendor price, so the values are independent and knapsack fits.
4. **The rest of the week** is planned day by day (value per shop hour), knowing those vendor cars are back Tuesday.

The deferred jobs' cost isn't saved, it moves to next week; the table shows it separately.

In [ ]:
budget = cfg.vendor.weekly_budget
deferred = {j.work_order_id for j in inputs.jobs.values() if not j.in_house and j.vehicle_class not in short_classes}
deferred_cost = sum(inputs.jobs[w].vendor_price for w in deferred)

# --- Cars each short class needs back by the first short morning -------------------
first_morning = shop_days[0] + READY_DELAY
in_repair_by_class = Counter(j.vehicle_class for j in inputs.jobs.values() if j.in_house)
need_first = {c: min(gap_before_repairs[(c, first_morning)], in_repair_by_class[c]) for c in short_classes}
needed = []
for c, n in need_first.items():
    jobs_c = sorted((j for j in inputs.jobs.values() if j.in_house and j.vehicle_class == c),
                    key=lambda j: (j.vendor_price, j.shop_hours, j.opened_ts))   # cheapest to cover first
    needed += jobs_c[:max(n, 0)]

# --- Monday's shop hours as a knapsack: keep the jobs that are most expensive to outsource ----
def monday_value(j):
    late = max(0, (vendor_ready(j) - first_morning).days)          # vendor can't make the first morning
    return j.vendor_price + inputs.daily_rate[j.vehicle_class] * late

items = [(j.work_order_id, round(j.shop_hours * 4), monday_value(j)) for j in needed]   # weights in quarter-hours
monday_shop = set(knapsack(items, int(FULL_DAY * 4)))
send_now = {j.work_order_id for j in needed} - monday_shop

def cover_pick(waiting, st):
    first = [j for j in waiting if j.work_order_id in monday_shop]
    if first:
        return first[0]                                            # all fit in Monday's hours
    return max(waiting, key=lambda j: score_value_per_hour(j, st))

name = "Defer spare vendor jobs + cover short classes"
plans[name], weeks[name] = build(cover_pick, vendor=send_now, defer=deferred)
# same choices but still sending the spare-class vendor jobs this week, to show the budget problem
plans[name + " (no defer)"], weeks[name + " (no defer)"] = build(cover_pick, vendor=send_now)
plans["B. Fewest short, within budget"], weeks["B. Fewest short, within budget"] = options["B. Fewest short, within budget"][:2]
plans["Value per shop hour"], weeks["Value per shop hour"] = build(by_score(score_value_per_hour))

print("1. Cars needed by", first_morning, "per short class:", need_first)
print(f"   Deferred to next week: {sorted(deferred)} (${deferred_cost:,.0f})")
print()
print("2. Monday: shop vs vendor for the cars needed first")
rows = [{"order": j.work_order_id, "class": j.vehicle_class, "job": j.job_type, "shop hours": j.shop_hours,
         "vendor price": j.vendor_price, "value of doing it in the shop": monday_value(j),
         "decision": "SHOP Monday" if j.work_order_id in monday_shop else "VENDOR Monday"} for j in needed]
display(pd.DataFrame(rows).sort_values(["decision", "class"]))
print(f"   Shop Monday: {sum(inputs.jobs[w].shop_hours for w in monday_shop):g} of {FULL_DAY:g} h, "
      f"vendor: {len(send_now)} jobs, ${sum(inputs.jobs[w].vendor_price for w in send_now):,.0f}")

compare = ["Oldest first (baseline)", "Value per shop hour", "B. Fewest short, within budget", name + " (no defer)", name]
print()
print("3. Cost of the week")
costs = cost_table(compare)
costs.loc["Budget left"] = [weeks[n].costs.budget_left for n in compare]
costs.loc["Vendor work moved to next week"] = [deferred_cost if n == name else 0 for n in compare]
display(costs.round(0))

print()
print(f"4. Cars short per day: {name}")
display(shortage_grid(weeks[name]).loc[sorted(short_classes)])
print()
print("5. Lost revenue by class")
display(lost_by_class(compare))

# The four options for the manager
| | Goal | Vendor-only jobs in spare classes |
|---|---|---|
| **1a** | Maximize profit (lowest week cost) | Deferred to next week |
| **1b** | Maximize profit | Sent this week |
| **2a** | Fewest cars short, using the budget | Deferred to next week |
| **2b** | Fewest cars short, using the budget | Sent this week (in case the vendor's turnaround is a concern) |

- **1a / 1b**: best shop plan (value per shop hour, day by day), then send a job to the vendor only if it lowers the week's cost, re-planning after each send.
- **2a / 2b**: strict cover plan: the cars each short class needs by the first short morning; Monday's shop hours chosen by knapsack, with jobs the vendor can't return in time (e.g. AC) kept in the shop; the rest go to the vendor Monday. If that exceeds the budget, drop sends one at a time (the one whose removal adds the least shortage per dollar saved) until it fits, then try swapping dropped and kept sends to find a cheaper plan with no more cars short.

Deferred work isn't saved: it moves to next week, shown on its own row.

In [30]:
def cover_plan(defer, strict=True):
    """Cover the first short morning: Monday shop hours by knapsack, the rest to the vendor; trim to budget."""
    penalty = 1e6                                                  # strict: avoid a late car whenever possible
    def value(j):
        late = max(0, (vendor_ready(j) - first_morning).days)
        return j.vendor_price + (penalty if strict and late else inputs.daily_rate[j.vehicle_class] * late)
    shop_monday = set(knapsack([(j.work_order_id, round(j.shop_hours * 4), value(j)) for j in needed], int(FULL_DAY * 4)))
    send = {j.work_order_id for j in needed} - shop_monday

    def pick(waiting, st):
        first = [j for j in waiting if j.work_order_id in shop_monday]
        return first[0] if first else max(waiting, key=lambda j: score_value_per_hour(j, st))

    plan, week = build(pick, vendor=send, defer=defer)
    trimmed = []
    while week.costs.budget_left < 0 and send:                     # over budget: drop the least useful send
        best = None
        for wo in sorted(send):
            p2, w2 = build(pick, vendor=send - {wo}, defer=defer)
            harm = (w2.costs.lost_revenue - week.costs.lost_revenue) / inputs.jobs[wo].vendor_price
            key = (harm, w2.costs.total, wo)        # equal harm: keep the cheaper week
            if best is None or key < best[0]:
                best = (key, wo, p2, w2)
        _, wo, plan, week = best
        send = send - {wo}
        trimmed.append(wo)

    # Clean-up pass: the trim is greedy, so try swapping a dropped send for a kept one;
    # keep any swap that is no worse on cars short, cheaper, and within budget.
    score = lambda w: (sum(c.short for c in w.shortages), w.costs.total)
    improved = True
    while improved and trimmed:
        improved = False
        for d_wo in sorted(trimmed):
            for k_wo in sorted(send):
                trial = (send - {k_wo}) | {d_wo}
                p2, w2 = build(pick, vendor=trial, defer=defer)
                if w2.costs.budget_left >= 0 and score(w2) < score(week):
                    plan, week, send = p2, w2, trial
                    trimmed = [x for x in trimmed if x != d_wo] + [k_wo]
                    improved = True
                    break
            if improved:
                break
    return plan, week, shop_monday, send, trimmed

value_pick = by_score(score_value_per_hour)
four = {}
p, w, s = vendor_option(value_pick, "savings", defer=deferred)
four["1a. Max profit, vendor-only next week"] = dict(plan=p, week=w, sends=[x["order"] for x in s], deferred=deferred)
p, w, s = vendor_option(value_pick, "savings")
four["1b. Max profit, vendor-only this week"] = dict(plan=p, week=w, sends=[x["order"] for x in s], deferred=set())
p, w, mon, send, trimmed = cover_plan(deferred)
four["2a. Fewest short, vendor-only next week"] = dict(plan=p, week=w, sends=sorted(send), deferred=deferred, trimmed=trimmed)
p, w, mon, send, trimmed = cover_plan(set())
four["2b. Fewest short, vendor-only this week"] = dict(plan=p, week=w, sends=sorted(send), deferred=set(), trimmed=trimmed)

for n, o in four.items():
    plans[n], weeks[n] = o["plan"], o["week"]

def moved(o):
    return sum(inputs.jobs[x].vendor_price for x in o["deferred"])

print("1. The four options")
summary = pd.DataFrame({n: {
    "Extra jobs sent to vendor": len(o["sends"]),
    "Shop": o["week"].costs.shop,
    "Vendor (this week)": o["week"].costs.vendor,
    "Lost revenue": o["week"].costs.lost_revenue,
    "Week cost (this week)": o["week"].costs.total,
    "Vendor work moved to next week": moved(o),
    "This week + moved": o["week"].costs.total + moved(o),
    "Budget left": o["week"].costs.budget_left,
    "Car-days short": sum(c.short for c in o["week"].shortages),
} for n, o in four.items()})
summary.insert(0, "Oldest first (today)", [0, weeks["Oldest first (baseline)"].costs.shop, weeks["Oldest first (baseline)"].costs.vendor,
                                           weeks["Oldest first (baseline)"].costs.lost_revenue, weeks["Oldest first (baseline)"].costs.total,
                                           0, weeks["Oldest first (baseline)"].costs.total, weeks["Oldest first (baseline)"].costs.budget_left,
                                           sum(c.short for c in weeks["Oldest first (baseline)"].shortages)])
display(summary.round(0))

for n, o in four.items():
    print()
    print(f"2. {n}")
    print(f"   Extra vendor sends ({len(o['sends'])}): {', '.join(job(x) for x in o['sends']) or 'none'}")
    if o.get("trimmed"):
        print(f"   Dropped to fit the budget: {', '.join(job(x) for x in o['trimmed'])}")
    if o["deferred"]:
        print(f"   Deferred to next week: {', '.join(job(x) for x in sorted(o['deferred']))}")
    display(shortage_grid(o["week"]).loc[sorted(short_classes)])

NameError: name 'deferred' is not defined

# Classes closest to meeting demand first (day by day)
Day by day: before every pick, the engine knows the shop clock and which cars are already coming back, and recalculates which jobs would still remove a short car.

**Rule:** jobs that would still remove a short car first, from the class with the **smallest remaining shortage** (given the cars already coming back), then shortest job, then oldest.

In [26]:
def score_closest(j, st):
    covered = st.covered(j)
    if not covered:
        return (False, 0, 0, -j.opened_ts.timestamp(), -wo_num(j))
    remaining_need = max(st.short_on(j.vehicle_class, d) for d in covered)   # how short the class still is
    return (True, -remaining_need, -j.shop_hours, -j.opened_ts.timestamp(), -wo_num(j))

name = "Closest to demand first"
plans[name], weeks[name] = build(by_score(score_closest))
compare = ["Oldest first (baseline)", "Short classes first", "Short first + shortest job",
           "Short first + most profitable", "Net-profit order", name]

print("1. Cost of the week")
costs = cost_table(compare)
display(costs.assign(**{"vs net-profit order": costs[name] - costs["Net-profit order"]}).round(0))

print()
print(f"2. Cars short per day: {name}")
display(shortage_grid(weeks[name]).loc[sorted(short_classes)])

print()
print("3. Lost revenue by class")
display(lost_by_class(compare))

print()
print("4. Short-class cars back in service by each morning")
display(cars_back_table(compare))

print()
print(f"5. Monday's 12 hours: {name}")
monday_work(name)

1. Cost of the week


,Oldest first (baseline),Short classes first,Short first + shortest job,Short first + most profitable,Net-profit order,Closest to demand first,vs net-profit order
Shop,12685.0,12410.0,12630.0,12630.0,12630.0,12630.0,0.0
Vendor,8780.0,8780.0,8780.0,8780.0,8780.0,8780.0,0.0
Lost revenue,8356.0,3880.0,2778.0,2628.0,2378.0,3526.0,1148.0
TOTAL,29821.0,25070.0,24188.0,24038.0,23788.0,24936.0,1148.0
Car-days short,97.0,43.0,32.0,36.0,28.0,43.0,15.0



2. Cars short per day: Closest to demand first


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,0,0,0,0,0,0
minivan,0,0,0,0,0,0,0
premium suv,0,0,0,0,0,0,0
suv,0,16,16,7,2,1,1



3. Lost revenue by class


,Oldest first (baseline),Short classes first,Short first + shortest job,Short first + most profitable,Net-profit order,Closest to demand first
midsize,550.0,0.0,0.0,660.0,0.0,0.0
minivan,552.0,276.0,92.0,0.0,92.0,0.0
premium suv,2006.0,1062.0,472.0,0.0,236.0,0.0
suv,5248.0,2542.0,2214.0,1968.0,2050.0,3526.0



4. Short-class cars back in service by each morning


,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
Oldest first (baseline),0,3,8,12,16,20,20
Short classes first,0,9,17,24,29,32,32
Short first + shortest job,0,14,22,26,31,32,32
Short first + most profitable,0,10,17,21,31,32,32
Net-profit order,0,13,20,25,31,32,32
Closest to demand first,0,10,17,26,31,32,32



5. Monday's 12 hours: Closest to demand first
   #1  WO-014 (minivan, oil change)              0.75 h   ready 2026-10-13
   #2  WO-028 (minivan, tires)                    1.5 h   ready 2026-10-13
   #3  WO-006 (premium suv, oil change)          0.75 h   ready 2026-10-13
   #4  WO-013 (premium suv, oil change)          0.75 h   ready 2026-10-13
   #5  WO-044 (premium suv, battery)              1.5 h   ready 2026-10-13
   #6  WO-024 (premium suv, tires)                1.5 h   ready 2026-10-13
   #7  WO-054 (premium suv, ac repair)            3.0 h   ready 2026-10-13
   #8  WO-016 (midsize, oil change)              0.75 h   ready 2026-10-13
   #9  WO-004 (midsize, oil change)              0.75 h   ready 2026-10-13
   #10 WO-011 (midsize, oil change)              0.75 h   ready 2026-10-13


# Most value per shop hour (day by day)
Day by day: before every pick, the engine knows the shop clock and which cars are already coming back, and recalculates which jobs would still remove a short car.

**Rule:** value covered = daily rate × the short days the car would actually cover if started now; pick the job with the most **value per shop hour** (ties: oldest). A class stops getting priority once it's covered, and a class whose shortage builds later gets cars later. The table compares every shop ordering.

In [28]:
pick_log = []
def logged_value_per_hour(waiting, st):
    j = max(waiting, key=lambda j: score_value_per_hour(j, st))
    covered = st.covered(j)
    value = inputs.daily_rate[j.vehicle_class] * len(covered)
    pick_log.append({"day": st.today, "order": j.work_order_id, "class": j.vehicle_class, "job": j.job_type,
                     "hours": j.shop_hours, "ready": st.ready(j) or "next week",
                     "short days covered": len(covered), "value covered": value,
                     "value per hour": round(value / j.shop_hours, 1)})
    return j

name = "Value per shop hour"
plans[name], weeks[name] = build(logged_value_per_hour)
compare = ["Oldest first (baseline)", "Short classes first", "Short first + shortest job", "Short first + most profitable",
           "Most profitable + best fit", "Net-profit order", "Closest to demand first", name]

print("1. Cost of the week: all shop orderings (no extra vendor sends)")
costs = cost_table(compare)
display(costs.assign(**{"vs baseline": costs[name] - costs["Oldest first (baseline)"],
                        "vs net-profit": costs[name] - costs["Net-profit order"]}).round(0))

print()
print("2. The picks, in order, with what each one covers (first 30)")
display(pd.DataFrame(pick_log).head(30))

print()
print(f"3. Cars short per day: {name}")
display(shortage_grid(weeks[name]).loc[sorted(short_classes)])

print()
print("4. Lost revenue by class")
display(lost_by_class(compare))

print()
print("5. Short-class cars back in service by each morning")
display(cars_back_table(compare))

1. Cost of the week: all shop orderings (no extra vendor sends)


,Oldest first (baseline),Short classes first,Short first + shortest job,Short first + most profitable,Most profitable + best fit,Net-profit order,Closest to demand first,Value per shop hour,vs baseline,vs net-profit
Shop,12685.0,12410.0,12630.0,12630.0,12096.0,12630.0,12630.0,12630.0,-55.0,0.0
Vendor,8780.0,8780.0,8780.0,8780.0,8780.0,8780.0,8780.0,8780.0,0.0,0.0
Lost revenue,8356.0,3880.0,2778.0,2628.0,2573.0,2378.0,3526.0,2332.0,-6024.0,-46.0
TOTAL,29821.0,25070.0,24188.0,24038.0,23449.0,23788.0,24936.0,23742.0,-6079.0,-46.0
Car-days short,97.0,43.0,32.0,36.0,35.0,28.0,43.0,28.0,-69.0,0.0



2. The picks, in order, with what each one covers (first 30)


,day,order,class,job,hours,ready,short days covered,value covered,value per hour
0,2026-10-12,WO-006,premium suv,oil change,0.75,2026-10-13,6,708.0,944.0
1,2026-10-12,WO-013,premium suv,oil change,0.75,2026-10-13,6,708.0,944.0
2,2026-10-12,WO-014,minivan,oil change,0.75,2026-10-13,6,552.0,736.0
3,2026-10-12,WO-007,suv,oil change,0.75,2026-10-13,6,492.0,656.0
4,2026-10-12,WO-002,suv,oil change,0.75,2026-10-13,6,492.0,656.0
5,2026-10-12,WO-012,suv,oil change,0.75,2026-10-13,6,492.0,656.0
6,2026-10-12,WO-010,suv,oil change,0.75,2026-10-13,6,492.0,656.0
7,2026-10-12,WO-044,premium suv,battery,1.50,2026-10-13,6,708.0,472.0
8,2026-10-12,WO-024,premium suv,tires,1.50,2026-10-13,6,708.0,472.0
9,2026-10-12,WO-016,midsize,oil change,0.75,2026-10-13,6,330.0,440.0



3. Cars short per day: Value per shop hour


date,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
vehicle_class,,,,,,,
midsize,0,0,0,0,0,0,0
minivan,0,0,0,0,0,0,0
premium suv,0,1,0,0,0,0,0
suv,0,12,8,4,1,1,1



4. Lost revenue by class


,Oldest first (baseline),Short classes first,Short first + shortest job,Short first + most profitable,Most profitable + best fit,Net-profit order,Closest to demand first,Value per shop hour
midsize,550.0,0.0,0.0,660.0,605.0,0.0,0.0,0.0
minivan,552.0,276.0,92.0,0.0,0.0,92.0,0.0,0.0
premium suv,2006.0,1062.0,472.0,0.0,0.0,236.0,0.0,118.0
suv,5248.0,2542.0,2214.0,1968.0,1968.0,2050.0,3526.0,2214.0



5. Short-class cars back in service by each morning


,2026-10-12,2026-10-13,2026-10-14,2026-10-15,2026-10-16,2026-10-17,2026-10-18
Oldest first (baseline),0,3,8,12,16,20,20
Short classes first,0,9,17,24,29,32,32
Short first + shortest job,0,14,22,26,31,32,32
Short first + most profitable,0,10,17,21,31,32,32
Most profitable + best fit,0,10,17,22,30,32,32
Net-profit order,0,13,20,25,31,32,32
Closest to demand first,0,10,17,26,31,32,32
Value per shop hour,0,13,21,26,31,32,32
